# 04 - The whole DOA estimator on the FPGA (Tier 3)

`scm_unitary` -> `jacobi_evd` -> `music_doa`, chained by AXI-Stream: the FPGA receives the frame
and returns the two DOA estimates (Q16.16 degrees) followed by the MUSIC denominator d(theta)
on the 181-point grid. The ARM only scales and packs the frame.

1. Bit-exact check of the board output against the integer models (SCM + Jacobi + MUSIC)
2. DOA directly from the FPGA, uncorrelated and coherent sources, with the hardware spectrum
3. Monte Carlo through the board: RMSE and resolution probability
4. Host-observed timing: FPGA vs NumPy on the ARM
5. PL latency in clock cycles, measured on the board with the AXI Timer
6. Throughput with many frames per DMA transfer (run this test last)

Upload `tier3_overlay.bit` and `tier3_overlay.hwh` into the same folder as this notebook.

In [ ]:
import os, time, math
import numpy as np
import pynq
from pynq import Overlay, allocate
print("PYNQ", pynq.__version__, "| numpy", np.__version__)

ol = Overlay(os.path.abspath("tier3_overlay.bit"))
print("IP blocks:", list(ol.ip_dict.keys()))
dma = ol.axi_dma_0
scm_ip = ol.scm_unitary_0
jac_ip = ol.jacobi_evd_0
mus_ip = ol.music_doa_0
tmr = ol.axi_timer_0

N, M = 8, 128
IN_FRAC, OUT_FRAC = 15, 27
A_FRAC, CS_FRAC = 25, 30
G, K = 181, 2
N_OUT = K + G
GRID = np.arange(-90.0, 90.5, 1.0)

In [ ]:
# ---- Bit-exact integer model of the IP (same as python/scm_bitexact.py) ----
def quantize_input(x):
    q = lambda v: np.clip(np.floor(v * 2**IN_FRAC + 0.5), -2**15, 2**15 - 1).astype(np.int64)
    return q(x.real), q(x.imag)

def scm_model_raw(xr, xi):
    h = N // 2
    a = xr[:h] - xr[::-1][:h]
    b = xi[:h] - xi[::-1][:h]
    yr = np.vstack([xr[:h] + xr[::-1][:h], b])
    yi = np.vstack([xi[:h] + xi[::-1][:h], -a])
    s = yr @ yr.T + yi @ yi.T
    shift = 2 * IN_FRAC + 8 - OUT_FRAC
    return np.clip((s + (1 << (shift - 1))) >> shift, -2**31, 2**31 - 1)

def pack(xr, xi):
    # one complex sample per word, [15:0] re, [31:16] im, snapshot-major
    return (xr.T.astype(np.uint16).astype(np.uint32)
            | (xi.T.astype(np.uint16).astype(np.uint32) << 16)).ravel()

in_buf = allocate(shape=(N * M,), dtype=np.uint32)
out_buf = allocate(shape=(N_OUT,), dtype=np.uint32)

def tier3_fpga_raw(words):
    """Frame -> (2 angles raw Q16.16 deg, d raw Q5.27), computed on the FPGA."""
    in_buf[:] = words
    dma.recvchannel.transfer(out_buf)
    mus_ip.write(0x00, 0x01)              # ap_start (each kernel waits for its input stream)
    jac_ip.write(0x00, 0x01)
    scm_ip.write(0x00, 0x01)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    o = np.array(out_buf).view(np.int32).astype(np.int64)
    return o[:K], o[K:]

def normalize(x):
    return x / (1.001 * max(np.abs(x.real).max(), np.abs(x.imag).max(), 1e-12))

In [ ]:
# ---- Bit-exact integer model of jacobi_evd (same as python/jacobi_bitexact.py) ----
ANG_FRAC, N_CORDIC, NORM_BITS = 28, 24, 30
ATAN = [int(round(math.atan(2.0 ** -i) * 2 ** ANG_FRAC)) for i in range(N_CORDIC)]
KINV = 1.0
for i in range(N_CORDIC):
    KINV /= math.sqrt(1.0 + 2.0 ** (-2 * i))
KINV = int(round(KINV * 2 ** CS_FRAC))
PAIRS = [(p, q) for p in range(N - 1) for q in range(p + 1, N)]

def rnd(x, sh):
    return (x + (1 << (sh - 1))) >> sh

def vectoring(x, y):
    m = max(abs(x), abs(y))
    if m == 0:
        return 0
    sh = NORM_BITS - m.bit_length()
    if sh > 0:
        x, y = x << sh, y << sh
    elif sh < 0:
        x, y = x >> -sh, y >> -sh
    z = 0
    for i in range(N_CORDIC):
        if y >= 0:
            x, y, z = x + (y >> i), y - (x >> i), z + ATAN[i]
        else:
            x, y, z = x - (y >> i), y + (x >> i), z - ATAN[i]
    return z

def rotation(th):
    x, y, z = KINV, 0, th
    for i in range(N_CORDIC):
        if z >= 0:
            x, y, z = x - (y >> i), y + (x >> i), z - ATAN[i]
        else:
            x, y, z = x + (y >> i), y - (x >> i), z + ATAN[i]
    return x, y

def jacobi_model_raw(r_raw, n_sweeps=4):
    a = [[rnd(int(v), 2) for v in row] for row in r_raw]
    v = [[(1 << CS_FRAC) if i == j else 0 for j in range(N)] for i in range(N)]
    for _ in range(n_sweeps):
        for p, q in PAIRS:
            app, aqq, apq = a[p][p], a[q][q], a[p][q]
            x, y = aqq - app, 2 * apq
            if x < 0:
                x, y = -x, -y
            c, s = rotation(vectoring(x, y) >> 1)
            c2, s2, cs = rnd(c * c, 30), rnd(s * s, 30), rnd(c * s, 30)
            for k in range(N):
                if k != p and k != q:
                    akp, akq = a[k][p], a[k][q]
                    a[k][p] = a[p][k] = rnd(c * akp - s * akq, 30)
                    a[k][q] = a[q][k] = rnd(s * akp + c * akq, 30)
                vkp, vkq = v[k][p], v[k][q]
                v[k][p] = rnd(c * vkp - s * vkq, 30)
                v[k][q] = rnd(s * vkp + c * vkq, 30)
            a[p][p] = rnd(c2 * app - 2 * cs * apq + s2 * aqq, 30)
            a[q][q] = rnd(s2 * app + 2 * cs * apq + c2 * aqq, 30)
            a[p][q] = a[q][p] = 0
    return np.array([a[i][i] for i in range(N)], dtype=np.int64), np.array(v, dtype=np.int64)

def to_float(d_raw, v_raw):
    d = d_raw * 2.0 ** -A_FRAC
    order = np.argsort(d)[::-1]
    return d[order], (v_raw * 2.0 ** -CS_FRAC)[:, order]

In [ ]:
# ---- Bit-exact integer model of music_doa (same as python/music_bitexact.py) ----
# Steering ROM, raw Q2.22, generated by python/gen_music_vectors.py (identical to hls/music/music_rom.h)
B_ROM = np.array([
    [0,-9934,-39731,-89381,-158858,-248116,-357076,-485609,-633523,-800538,-986264,-1190174,-1411578,-1649591,-1903104,-2170749,-2450871,-2741499,-3040313,-3344627,-3651365,-3957052,-4257807,-4549349,-4827018,-5085800,-5320374,-5525175,-5694469,-5822455,-5903373,-5931641,-5902007,-5809707,-5650647,-5421591,-5120345,-4745951,-4298866,-3781127,-3196493,-2550557,-1850814,-1106682,-329467,467737,1270217,2061882,2825577,3543471,4197510,4769924,5243792,5603624,5835966,5929987,5878040,5676155,5324459,4827477,4194304,3438625,2578567,1636377,637919,-387999,-1410462,-2397417,-3316747,-4137425,-4830683,-5371161,-5737992,-5915756,-5895267,-5674149,-5257160,-4656242,-3890285,-2984599,-1970100,-882252,240214,1356744,2426517,3410037,4270706,4976291,5500239,5822759,5931642,5822759,5500239,4976291,4270706,3410037,2426517,1356744,240214,-882252,-1970100,-2984599,-3890285,-4656242,-5257160,-5674149,-5895267,-5915756,-5737992,-5371161,-4830683,-4137425,-3316747,-2397417,-1410462,-387999,637919,1636377,2578567,3438625,4194304,4827477,5324459,5676155,5878040,5929987,5835966,5603624,5243792,4769924,4197510,3543471,2825577,2061882,1270217,467737,-329467,-1106682,-1850814,-2550557,-3196493,-3781127,-4298866,-4745951,-5120345,-5421591,-5650647,-5809707,-5902007,-5931641,-5903373,-5822455,-5694469,-5525175,-5320374,-5085800,-4827018,-4549349,-4257807,-3957052,-3651365,-3344627,-3040313,-2741499,-2450871,-2170749,-1903104,-1649591,-1411578,-1190174,-986264,-800538,-633523,-485609,-357076,-248116,-158858,-89381,-39731,-9934,0],
    [0,7095,28379,63845,113477,177251,255130,347054,452940,572670,706083,852967,1013047,1185974,1371314,1568533,1776987,1995908,2224390,2461379,2705660,2955846,3210370,3467476,3725218,3981452,4233842,4479862,4716806,4941801,5151824,5343730,5514276,5660157,5778052,5864663,5916773,5931301,5905364,5836344,5721955,5560317,5350026,5090221,4780656,4421762,4014701,3561416,3064668,2528058,1956039,1353909,727783,84554,-568176,-1222177,-1868701,-2498601,-3102481,-3670850,-4194304,-4663704,-5070375,-5406300,-5664313,-5838295,-5923341,-5915931,-5814063,-5617368,-5327191,-4946636,-4480580,-3935638,-3320091,-2643773,-1917919,-1154969,-368345,427810,1218907,1990216,2727176,3415703,4042506,4595379,5063474,5437557,5710209,5876006,5931642,5876006,5710209,5437557,5063474,4595379,4042506,3415703,2727176,1990216,1218907,427810,-368345,-1154969,-1917919,-2643773,-3320091,-3935638,-4480580,-4946636,-5327191,-5617368,-5814063,-5915931,-5923341,-5838295,-5664313,-5406300,-5070375,-4663704,-4194304,-3670850,-3102481,-2498601,-1868701,-1222177,-568176,84554,727783,1353909,1956039,2528058,3064668,3561416,4014701,4421762,4780656,5090221,5350026,5560317,5721955,5836344,5905364,5931301,5916773,5864663,5778052,5660157,5514276,5343730,5151824,4941801,4716806,4479862,4233842,3981452,3725218,3467476,3210370,2955846,2705660,2461379,2224390,1995908,1776987,1568533,1371314,1185974,1013047,852967,706083,572670,452940,347054,255130,177251,113477,63845,28379,7095,0],
    [0,-4257,-17028,-38307,-68089,-106361,-153108,-208309,-271934,-343945,-424294,-512919,-609744,-714673,-827591,-948361,-1076817,-1212766,-1355981,-1506201,-1663124,-1826408,-1995664,-2170456,-2350296,-2534643,-2722900,-2914412,-3108463,-3304277,-3501017,-3697784,-3893617,-4087497,-4278345,-4465029,-4646364,-4821118,-4988019,-5145760,-5293005,-5428399,-5550579,-5658180,-5749850,-5824258,-5880112,-5916167,-5931242,-5924232,-5894127,-5840019,-5761126,-5656797,-5526534,-5370000,-5187031,-4977654,-4742087,-4480758,-4194304,-3883578,-3549654,-3193827,-2817608,-2422725,-2011112,-1584903,-1146419,-698151,-242748,217005,678209,1137871,1592934,2040299,2476859,2899520,3305237,3691040,4054068,4391592,4701047,4980059,5226466,5438348,5614041,5752158,5851605,5911593,5931642,5911593,5851605,5752158,5614041,5438348,5226466,4980059,4701047,4391592,4054068,3691040,3305237,2899520,2476859,2040299,1592934,1137871,678209,217005,-242748,-698151,-1146419,-1584903,-2011112,-2422725,-2817608,-3193827,-3549654,-3883578,-4194304,-4480758,-4742087,-4977654,-5187031,-5370000,-5526534,-5656797,-5761126,-5840019,-5894127,-5924232,-5931242,-5916167,-5880112,-5824258,-5749850,-5658180,-5550579,-5428399,-5293005,-5145760,-4988019,-4821118,-4646364,-4465029,-4278345,-4087497,-3893617,-3697784,-3501017,-3304277,-3108463,-2914412,-2722900,-2534643,-2350296,-2170456,-1995664,-1826408,-1663124,-1506201,-1355981,-1212766,-1076817,-948361,-827591,-714673,-609744,-512919,-424294,-343945,-271934,-208309,-153108,-106361,-68089,-38307,-17028,-4257,0],
    [0,1419,5676,12769,22697,35455,51041,69449,90673,114706,141539,171163,203568,238740,276666,317331,360718,406807,455577,507006,561068,617736,676979,738765,803058,869820,939010,1010582,1084490,1160681,1239101,1319692,1402392,1487134,1573848,1662461,1752894,1845065,1938888,2034272,2131122,2229338,2328819,2429455,2531135,2633743,2737159,2841260,2945917,3050999,3156372,3261897,3367433,3472836,3577958,3682651,3786762,3890139,3992625,4094066,4194304,4293181,4390539,4486220,4580067,4671925,4761637,4849051,4934015,5016382,5096006,5172745,5246461,5317019,5384291,5448152,5508484,5565173,5618112,5667201,5712346,5753460,5790464,5823287,5851865,5876141,5896069,5911609,5922731,5929413,5931642,5929413,5922731,5911609,5896069,5876141,5851865,5823287,5790464,5753460,5712346,5667201,5618112,5565173,5508484,5448152,5384291,5317019,5246461,5172745,5096006,5016382,4934015,4849051,4761637,4671925,4580067,4486220,4390539,4293181,4194304,4094066,3992625,3890139,3786762,3682651,3577958,3472836,3367433,3261897,3156372,3050999,2945917,2841260,2737159,2633743,2531135,2429455,2328819,2229338,2131122,2034272,1938888,1845065,1752894,1662461,1573848,1487134,1402392,1319692,1239101,1160681,1084490,1010582,939010,869820,803058,738765,676979,617736,561068,507006,455577,406807,360718,317331,276666,238740,203568,171163,141539,114706,90673,69449,51041,35455,22697,12769,5676,1419,0],
    [-5931642,-5931633,-5931509,-5930968,-5929514,-5926450,-5920884,-5911730,-5897713,-5877373,-5849073,-5811012,-5761234,-5697650,-5618057,-5520165,-5401630,-5260091,-5093218,-4898759,-4674602,-4418836,-4129825,-3806283,-3447356,-3052705,-2622593,-2157966,-1660540,-1132869,-578415,-1594,592189,1196529,1804039,2406392,2994401,3558134,4087068,4570279,4996679,5355281,5635500,5827489,5922485,5913171,5794042,5561746,5215409,4756909,4191096,3525932,2772547,1945191,1061070,140075,-795627,-1722103,-2614289,-3446714,-4194304,-4833243,-5341851,-5701460,-5897239,-5918938,-5761507,-5425566,-4917678,-4250422,-3442220,-2516943,-1503269,-433821,655892,1728699,2747116,3674750,4477728,5126065,5594915,5865663,5926776,5774393,5412614,4853454,4116484,3228142,2220754,1131304,0,-1131304,-2220754,-3228142,-4116484,-4853454,-5412614,-5774393,-5926776,-5865663,-5594915,-5126065,-4477728,-3674750,-2747116,-1728699,-655892,433821,1503269,2516943,3442220,4250422,4917678,5425566,5761507,5918938,5897239,5701460,5341851,4833243,4194304,3446714,2614289,1722103,795627,-140075,-1061070,-1945191,-2772547,-3525932,-4191096,-4756909,-5215409,-5561746,-5794042,-5913171,-5922485,-5827489,-5635500,-5355281,-4996679,-4570279,-4087068,-3558134,-2994401,-2406392,-1804039,-1196529,-592189,1594,578415,1132869,1660540,2157966,2622593,3052705,3447356,3806283,4129825,4418836,4674602,4898759,5093218,5260091,5401630,5520165,5618057,5697650,5761234,5811012,5849073,5877373,5897713,5911730,5920884,5926450,5929514,5930968,5931509,5931633,5931642],
    [5931642,5931637,5931574,5931298,5930556,5928993,5926152,5921480,5914323,5903933,5889467,5869993,5844494,5811870,5770950,5720496,5659213,5585761,5498769,5396849,5278615,5142698,4987775,4812586,4615964,4396864,4154390,3887828,3596681,3280698,2939911,2574669,2185666,1773976,1341077,888876,419724,-63566,-557718,-1058993,-1563203,-2065731,-2561561,-3045328,-3511367,-3953782,-4366526,-4743489,-5078600,-5365938,-5599847,-5775059,-5886825,-5931039,-5904367,-5804365,-5629594,-5379718,-5055590,-4659317,-4194304,-3665274,-3078257,-2440552,-1760661,-1048184,-313694,431428,1175176,1905137,2608719,3273403,3887001,4437919,4915422,5309881,5613017,5818111,5920194,5916194,5805053,5587791,5267531,4849468,4340797,3750582,3089595,2370095,1605581,810512,0,-810512,-1605581,-2370095,-3089595,-3750582,-4340797,-4849468,-5267531,-5587791,-5805053,-5916194,-5920194,-5818111,-5613017,-5309881,-4915422,-4437919,-3887001,-3273403,-2608719,-1905137,-1175176,-431428,313694,1048184,1760661,2440552,3078257,3665274,4194304,4659317,5055590,5379718,5629594,5804365,5904367,5931039,5886825,5775059,5599847,5365938,5078600,4743489,4366526,3953782,3511367,3045328,2561561,2065731,1563203,1058993,557718,63566,-419724,-888876,-1341077,-1773976,-2185666,-2574669,-2939911,-3280698,-3596681,-3887828,-4154390,-4396864,-4615964,-4812586,-4987775,-5142698,-5278615,-5396849,-5498769,-5585761,-5659213,-5720496,-5770950,-5811870,-5844494,-5869993,-5889467,-5903933,-5914323,-5921480,-5926152,-5928993,-5930556,-5931298,-5931574,-5931637,-5931642],
    [-5931642,-5931640,-5931617,-5931518,-5931251,-5930688,-5929665,-5927983,-5925405,-5921661,-5916447,-5909423,-5900219,-5888431,-5873625,-5855338,-5833081,-5806339,-5774572,-5737223,-5693715,-5643457,-5585848,-5520280,-5446144,-5362831,-5269743,-5166292,-5051914,-4926066,-4788241,-4637970,-4474831,-4298458,-4108544,-3904854,-3687232,-3455603,-3209990,-2950513,-2677401,-2390994,-2091756,-1780272,-1457258,-1123561,-780163,-428182,-68871,296383,666064,1038530,1412020,1784661,2154481,2519420,2877339,3226040,3563282,3886795,4194304,4483547,4752297,4998384,5219718,5414312,5580305,5715982,5819802,5890412,5926672,5927671,5892742,5821479,5713750,5569699,5389763,5174665,4925422,4643339,4330000,3987266,3617254,3222327,2805071,2368279,1914921,1448120,971126,487283,0,-487283,-971126,-1448120,-1914921,-2368279,-2805071,-3222327,-3617254,-3987266,-4330000,-4643339,-4925422,-5174665,-5389763,-5569699,-5713750,-5821479,-5892742,-5927671,-5926672,-5890412,-5819802,-5715982,-5580305,-5414312,-5219718,-4998384,-4752297,-4483547,-4194304,-3886795,-3563282,-3226040,-2877339,-2519420,-2154481,-1784661,-1412020,-1038530,-666064,-296383,68871,428182,780163,1123561,1457258,1780272,2091756,2390994,2677401,2950513,3209990,3455603,3687232,3904854,4108544,4298458,4474831,4637970,4788241,4926066,5051914,5166292,5269743,5362831,5446144,5520280,5585848,5643457,5693715,5737223,5774572,5806339,5833081,5855338,5873625,5888431,5900219,5909423,5916447,5921661,5925405,5927983,5929665,5930688,5931251,5931518,5931617,5931640,5931642],
    [5931642,5931641,5931639,5931628,5931598,5931536,5931422,5931235,5930949,5930532,5929953,5929172,5928147,5926835,5925186,5923147,5920663,5917675,5914121,5909934,5905047,5899388,5892883,5885457,5877029,5867520,5856845,5844921,5831660,5816974,5800776,5782974,5763477,5742195,5719036,5693909,5666722,5637385,5605808,5571904,5535584,5496765,5455362,5411296,5364488,5314863,5262350,5206882,5148393,5086824,5022120,4954230,4883110,4808720,4731024,4649995,4565611,4477856,4386720,4292201,4194304,4093039,3988426,3880490,3769264,3654790,3537116,3416296,3292395,3165483,3035637,2902944,2767494,2629388,2488731,2345636,2200223,2052614,1902943,1751345,1597961,1442939,1286427,1128583,969563,809531,648649,487086,325010,162591,0,-162591,-325010,-487086,-648649,-809531,-969563,-1128583,-1286427,-1442939,-1597961,-1751345,-1902943,-2052614,-2200223,-2345636,-2488731,-2629388,-2767494,-2902944,-3035637,-3165483,-3292395,-3416296,-3537116,-3654790,-3769264,-3880490,-3988426,-4093039,-4194304,-4292201,-4386720,-4477856,-4565611,-4649995,-4731024,-4808720,-4883110,-4954230,-5022120,-5086824,-5148393,-5206882,-5262350,-5314863,-5364488,-5411296,-5455362,-5496765,-5535584,-5571904,-5605808,-5637385,-5666722,-5693909,-5719036,-5742195,-5763477,-5782974,-5800776,-5816974,-5831660,-5844921,-5856845,-5867520,-5877029,-5885457,-5892883,-5899388,-5905047,-5909934,-5914121,-5917675,-5920663,-5923147,-5925186,-5926835,-5928147,-5929172,-5929953,-5930532,-5930949,-5931235,-5931422,-5931536,-5931598,-5931628,-5931639,-5931641,-5931642]
], dtype=np.int64)

def music_model_raw(d_eig, v):
    order = sorted(range(N), key=lambda i: (-int(d_eig[i]), i))[:K]
    d = []
    eight = 8 << 58
    for g in range(G):
        s = 0
        for k in order:
            acc = 0
            for n in range(N):
                acc += int(v[n, k]) * int(B_ROM[n, g])
            p = rnd(acc, 23)
            s += p * p
        d.append(rnd(eight - s, 31))
    minima, overall = [], []
    for g in range(G):
        overall = sorted(overall + [(d[g], g)])[:K]
        if 0 < g < G - 1 and d[g] < d[g - 1] and d[g] <= d[g + 1]:
            minima = sorted(minima + [(d[g], g)])[:K]
    chosen = [g for _, g in minima]
    for _, g in overall:
        if len(chosen) >= K:
            break
        if g not in chosen:
            chosen.append(g)
    angles = []
    for g in chosen:
        delta = 0
        if 0 < g < G - 1:
            y0, y1, y2 = d[g - 1], d[g], d[g + 1]
            den = y0 - 2 * y1 + y2
            if den > 0:
                num = y0 - y2
                q = (abs(num) << 16) // (2 * den)
                delta = q if num >= 0 else -q
        angles.append(((g - 90) << 16) + delta)
    return np.array(sorted(angles), dtype=np.int64), np.array(d, dtype=np.int64)

In [ ]:
# ---- Test 1: bit-exact on 20 random frames (the ARM models take ~0.1-0.3 s per frame) ----
rng = np.random.RandomState(1)
fails = 0
for t in range(20):
    x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
    xr, xi = quantize_input(x)
    a_hw, d_hw = tier3_fpga_raw(pack(xr, xi))
    a_ref, d_ref = music_model_raw(*jacobi_model_raw(scm_model_raw(xr, xi)))
    if not (np.array_equal(a_hw, a_ref) and np.array_equal(d_hw, d_ref)):
        fails += 1
        if fails <= 3:
            print("frame", t, "angles hw", a_hw, "ref", a_ref, "max |d diff|", np.abs(d_hw - d_ref).max())
print("bit-exact frames: %d / 20 %s" % (20 - fails, "PASS" if fails == 0 else "FAIL"))

In [ ]:
# ---- Test 2: DOA directly from the FPGA ----
pos = (np.arange(N) - (N - 1) / 2) * 0.5          # centred virtual ULA, d = lambda/2

def steering(theta_deg):
    return np.exp(1j * 2 * np.pi * np.outer(pos, np.sin(np.deg2rad(np.atleast_1d(theta_deg)))))

def simulate(theta, snr_db, coherent=False):
    k = len(theta)
    s = (rng.randn(k, M) + 1j * rng.randn(k, M)) / np.sqrt(2)
    if coherent:
        s = np.repeat(s[:1], k, axis=0) * np.exp(1j * rng.uniform(0, 2 * np.pi, (k, 1)))
    n = 10 ** (-snr_db / 20) * (rng.randn(N, M) + 1j * rng.randn(N, M)) / np.sqrt(2)
    return steering(theta) @ s + n

def fpga_doa(x):
    xr, xi = quantize_input(normalize(x))
    a, d = tier3_fpga_raw(pack(xr, xi))
    return a * 2.0 ** -16, d * 2.0 ** -27

true = np.array([-10.0, 5.0])
results = {}
for coherent in (False, True):
    est, d = fpga_doa(simulate(true, 10, coherent))
    results[coherent] = -10 * np.log10(np.maximum(d, 1e-9))
    print("%-14s true %s  ->  FPGA estimate %s deg" % ("coherent:" if coherent else "uncorrelated:", true, np.round(est, 3)))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for coherent, spec in results.items():
    plt.plot(GRID, spec - spec.max(), label="coherent" if coherent else "uncorrelated")
for t in true:
    plt.axvline(t, color="gray", ls=":")
plt.xlabel("angle (deg)"); plt.ylabel("MUSIC spectrum (dB)")
plt.title("MUSIC spectrum computed on the FPGA, N=8, M=128, SNR 10 dB")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ---- Test 3: Monte Carlo through the board (100 trials per point) ----
rng = np.random.RandomState(7)
print("%-13s %6s %10s %12s" % ("sources", "SNR", "RMSE(deg)", "P(resolved)"))
for coherent in (False, True):
    for snr in (0, 10, 20):
        err, ok = [], 0
        for _ in range(100):
            est, _ = fpga_doa(simulate(true, snr, coherent))
            e = np.sort(est) - true
            err.append(e)
            ok += int(np.all(np.abs(e) < 2.0))
        print("%-13s %4d dB %10.4f %12.2f" % ("coherent" if coherent else "uncorrelated", snr,
                                             np.sqrt(np.mean(np.square(err))), ok / 100.0))

In [ ]:
# ---- Test 4: host-observed time, FPGA vs NumPy on the ARM (median of 200) ----
x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
xr, xi = quantize_input(x)
words = pack(xr, xi)
xq = (xr + 1j * xi) * 2.0**-IN_FRAC
h = N // 2
I, J = np.eye(h), np.fliplr(np.eye(h))
Q = np.block([[I, 1j * I], [J, -1j * J]]) / np.sqrt(2)
B = B_ROM * 2.0 ** -22

def median_us(fn, reps=200):
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return 1e6 * np.median(t)

def numpy_doa():
    y = Q.conj().T @ xq
    w, v = np.linalg.eigh((y @ y.conj().T).real / M)
    d = 8.0 - np.sum((v[:, -K:].T @ B) ** 2, axis=0)
    mins = [g for g in range(1, G - 1) if d[g] < d[g - 1] and d[g] <= d[g + 1]]
    return sorted(mins, key=lambda g: d[g])[:K]

t_fpga = median_us(lambda: tier3_fpga_raw(words))
t_np = median_us(numpy_doa)
print("FPGA whole estimator (DMA + 3 IPs, host-observed): %9.1f us" % t_fpga)
print("NumPy float SCM + eigh + MUSIC + peaks, ARM      : %9.1f us" % t_np)
print("HLS estimate of the IPs alone: see hls/*/reports")

In [ ]:
# ---- Test 5: PL latency in clock cycles (AXI Timer, 100 MHz) ----
# Counter 0 captures when the DMA has delivered the whole frame to scm_unitary
# (mm2s_introut), counter 1 when the last output word is in DDR (s2mm_introut).
# Hold mode: each counter keeps its first capture until its interrupt bit is cleared.
MDT, CAPT, LOAD, ENT, TINT, ENALL = 0x1, 0x8, 0x20, 0x80, 0x100, 0x400
tmr.write(0x04, 0); tmr.write(0x14, 0)
tmr.write(0x00, LOAD); tmr.write(0x10, LOAD)
tmr.write(0x10, MDT | CAPT)
tmr.write(0x00, MDT | CAPT | ENALL)          # start both counters together

def arm_capture():
    for off in (0x00, 0x30):                  # DMA IOC interrupt enable (introut)
        dma.write(off, dma.read(off) | 0x1000)
    dma.write(0x04, 0x1000); dma.write(0x34, 0x1000)     # clear IOC -> introut low
    tmr.write(0x00, MDT | CAPT | ENT | TINT)               # clear capture flags
    tmr.write(0x10, MDT | CAPT | ENT | TINT)

def pl_cycles(words):
    arm_capture()
    tier3_fpga_raw(words)
    return (tmr.read(0x14) - tmr.read(0x04)) & 0xFFFFFFFF

c = np.array([pl_cycles(words) for _ in range(200)])
print("input accepted -> output in DDR: median %d cycles (min %d, max %d)" % (np.median(c), c.min(), c.max()))
print("+ 1024 cycles to stream the frame in: about %.1f us per frame in the PL" % ((np.median(c) + 1024) / 100.0))
print("compare: co-simulation latency of the three kernels, hls/*/reports")

In [ ]:
# ---- Test 6: throughput, many frames per DMA transfer (run last) ----
# The kernels run with auto-restart; music_doa sets TLAST only on every B-th frame,
# so one DMA transfer moves B frames in and B results out.
def reg_offset(ip, reg, default):
    try:
        return ol.ip_dict[ip]['registers'][reg]['address_offset']
    except Exception:
        return default

B = 100
xs = [normalize(rng.randn(N, M) + 1j * rng.randn(N, M)) for _ in range(B)]
qs = [quantize_input(x) for x in xs]
big_in = allocate(shape=(B * N * M,), dtype=np.uint32)
big_out = allocate(shape=(B * N_OUT,), dtype=np.uint32)
big_in[:] = np.concatenate([pack(xr, xi) for xr, xi in qs])

mus_ip.write(reg_offset('music_doa_0', 'batch', 0x10), B)
for ip in (mus_ip, jac_ip, scm_ip):
    ip.write(0x00, 0x81)                      # ap_start + auto_restart
arm_capture()
t0_host = time.perf_counter()
t0 = tmr.read(0x18)                           # live counter 1
dma.recvchannel.transfer(big_out)
dma.sendchannel.transfer(big_in)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_host = time.perf_counter() - t0_host
cycles = (tmr.read(0x14) - t0) & 0xFFFFFFFF

out = np.array(big_out).view(np.int32).astype(np.int64).reshape(B, N_OUT)
ok = 0
for f in range(3):                            # spot-check against the integer models
    a_ref, d_ref = music_model_raw(*jacobi_model_raw(scm_model_raw(*qs[f])))
    ok += int(np.array_equal(out[f, :K], a_ref) and np.array_equal(out[f, K:], d_ref))
print("spot check: %d / 3 frames bit-exact" % ok)
print("PL: %.0f cycles per frame -> %.1f us, %.0f frames/s" % (cycles / B, cycles / B / 100.0, 1e8 * B / cycles))
print("host-observed: %.1f us per frame (one transfer of %d frames)" % (1e6 * t_host / B, B))
print("Note: the kernels keep running in auto-restart; reload the overlay before other tests.")